# New York State of Energy - Renewable Energy by 2030

*Team: Muhammad Haseeb Anjum, Graham Haun, Melissa Marshall, Deval Mehta, Damar Shipp*

### Overview
According to the [New York State Energy Plan](https://energyplan.ny.gov/), the State of New York intends to reduce greenhouse gas emissions to 85% of their 1990 levels by 2050. To this end, the State must produce and maintain renewable energy infrastructure to gradually replace the existing carbon-based energy systems in place on a similar, if not accelerated, timescale. In particular, the various climatological zones of New York State are amenable to wind, solar, and hydroelectric power. In order to determine the best locations for each source of energy, we must consider a variety of factors, from typical weather to cost to land use agreements.

We employ clustering methods and time-series analysis to classify the state into these various climatological zones, so that we might simplify the process for the State Energy Planning Board to determine which land use agreements should be considered for each type of renewable energy source. We then perform a predictive time-series analysis to demonstrate that our proposed plan will continue to serve the State into the near future, in alignment with the state's benchmark goals in 2030.

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from tslearn.clustering import TimeSeriesKMeans

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

## Predictive and Time-Local Climate Modeling

### Load in the Weather Data

In [ ]:
weather = pd.read_csv('../data/new-york-weather.csv', low_memory=False)
weather.shape

### Inspect and Handle the Weather Data

In [ ]:
# Check for null values
weather.isnull().sum()

In [ ]:
# Check datatypes for the columns
weather.dtypes

The UV index columns (`uv_index_max` and `uv_index_clear_sky_max`) were
completely empty across all locations and dates — Open-Meteo did not return
values for these variables in the archive API. We drop them before proceeding.

In [ ]:
# Drop empty columns
weather = weather.drop(columns = ['uv_index_max','uv_index_clear_sky_max'])

In [ ]:
weather = weather.drop(weather[weather['daylight_duration'].astype(str).str.contains('day')].index)

In [ ]:
# Sort the DataFrame by date
weather = weather.sort_values('date')

In [ ]:
# Open-Meteo appends a UTC offset timestamp (05:00:00+00:00) to every date.
# Strip it and remove any resulting whitespace before converting to datetime.
weather['date'] = weather['date'].str.replace('05:00:00+00:00', '')
weather['date'] = weather['date'].str.strip()

In [ ]:
# Ensure feature columns are numeric
weather[['temperature_2m_max','temperature_2m_min',"daylight_duration","sunshine_duration","rain_sum",'showers_sum',
         "snowfall_sum","precipitation_hours","wind_speed_10m_max","wind_gusts_10m_max","latitude","longitude"]] = weather[['temperature_2m_max','temperature_2m_min',"daylight_duration","sunshine_duration","rain_sum",'showers_sum',
         "snowfall_sum","precipitation_hours","wind_speed_10m_max","wind_gusts_10m_max","latitude","longitude"]].apply(pd.to_numeric)

### Data Exploration

In [ ]:
weather['precipitation_total'] = weather['rain_sum'] + weather['snowfall_sum'] 
weather['location'] = list(zip(weather['latitude'],weather['longitude']))

In [ ]:
# Group by 'location'
grouped = weather.groupby('location')

# Drop duplicates based on 'date' for each group
weather = pd.concat([group.drop_duplicates(subset=['date']) for _, group in grouped], ignore_index=True)

# Display the first few rows of the cleaned DataFrame
weather.shape

In [ ]:
weather.info()

In [ ]:
weather.isnull().sum()

In [ ]:
weather = weather.dropna()
weather.head()

In [ ]:
weather.info() #this is after dropping those two nulls

In [ ]:
weather.isnull().sum() #nulls are removed

### Feature Relationship Visualizations

In [ ]:
thresholds = [10, 15, 20]  #  thresholds
frequencies = [(weather['wind_speed_10m_max'] > t).sum() for t in thresholds]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, frequencies, marker='o')
plt.title('Frequency of Days with Wind Speeds Above Thresholds')
plt.xlabel('Wind Speed Threshold (m/s)')
plt.ylabel('Frequency')
plt.xticks(thresholds)
plt.grid(True)
plt.savefig('../images/wind_speeds_by_threshold_surpassed.png', dpi = 300)   
plt.show();

In [ ]:
#looking at the feature relationships
sns.pairplot(weather[['temperature_2m_max', 'temperature_2m_min', 'rain_sum', 'wind_speed_10m_max']])
plt.savefig('../images/feature_pairplot.png', dpi = 300)
plt.show();

In [ ]:
#identifying highly correlated features for possible feature engineering. bypassing the string conversion erro
# only the numeric columns
numeric_weather = weather.select_dtypes(include=[np.number])

# Calculate the correlation matrix
corr = numeric_weather.corr()

# Create the heatmap
plt.figure(figsize=(12, 10))  
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title('Correlation Heatmap')
plt.savefig('../images/correlation_heatmap', dpi = 300)
plt.show();

In [ ]:
# only the numeric columns
numeric_weather = weather.select_dtypes(include=[np.number])

# Calculate the correlation matrix
corr = numeric_weather.corr()

print(corr)

#### Correlations Greater Than 0.5 or Less Than -0.5
Here are the correlations from above that exceed 0.5 or are below -0.5 and these indicates a strong positive or negative linear relationship. these are good to use for feature selection to improve clustering model

- temperature_2m_max and temperature_2m_min: 0.9436

- temperature_2m_max and daylight_duration: 0.7620

- temperature_2m_min and daylight_duration: 0.7173

- precipitation_total and rain_sum: 0.9696

- precipitation_hours and rain_sum: 0.6514

- wind_speed_above_20 and wind_speed_10m_max: 0.7886

- wind_speed_above_20 and wind_gusts_10m_max: 0.7064

- latitude and frequency_above_20: 0.9567

### Feature Engineering

In [ ]:
#thresholds
thresholds = [10, 15, 20] 

#store frequencies
wind_speed_frequencies = {}

# Calculate the frequency for each threshold
for threshold in thresholds:
    count_above_threshold = (weather['wind_speed_10m_max'] > threshold).sum()
    wind_speed_frequencies[f'wind_speed_above_{threshold}'] = count_above_threshold

# Converted to a df
wind_speed_frequency_df = pd.DataFrame.from_dict(wind_speed_frequencies, orient='index', columns=['Frequency'])

print(wind_speed_frequency_df)

We use the threshold of 20 m/s — the final value from the loop in the cell
above — to engineer the `wind_speed_above_20` and `frequency_above_20` columns.
20 m/s is the threshold that most meaningfully separates high-wind locations
from low-wind ones based on the frequency plot above.

In [ ]:
# Calculate a boolean column indicating if the wind speed exceeds the threshold
weather[f'wind_speed_above_{threshold}'] = weather['wind_speed_10m_max'] > threshold

# Convert boolean to integer (1 for True, 0 for False)
weather[f'wind_speed_above_{threshold}'] = weather[f'wind_speed_above_{threshold}'].astype(int)

# Added a frequency column by summing the boolean column
weather[f'frequency_above_{threshold}'] = weather[f'wind_speed_above_{threshold}'].cumsum()

print(weather.head())

In [ ]:
# adding a temperature range feature using two of the features that had good correlations
weather['temperature_range'] = weather['temperature_2m_max'] - weather['temperature_2m_min']

In [ ]:
# average temperature
weather['average_temperature'] = (weather['temperature_2m_max'] + weather['temperature_2m_min']) / 2

In [ ]:
#temperature vs daylight
weather['temp_daylight_interaction'] = weather['average_temperature'] * weather['daylight_duration']

In [ ]:
#wind speed index to rep  overall wind speed conditions
weather['wind_speed_index'] = weather['wind_speed_above_20'] * weather['wind_speed_10m_max']

### Exploratory Model: $k$-Means Clustering

We iterated on the k-means model to improve the silhouette score, which
measures cluster separation on a scale of -1 to 1 (higher is better):

- Initial score with `n_clusters=3`, `n_components=5`: **0.29**
- Reduced to `n_clusters=2`: **0.44**
- Added `wind_speed_frequency` feature: **0.407** (slight regression — kept feature for interpretability)
- Added `temperature_range` feature: **0.446**
- Added interaction/aggregation features: **0.559**
- Reduced to `n_components=2`: **0.575** (final)

A three-cluster solution (solar, wind, hydro) did not produce well-separated
clusters. The data supports two climatological zones. The pipeline below
reflects these final parameter choices.

In [ ]:
weather_km = weather.copy()
weather_km['date'] = pd.to_datetime(weather_km['date'])
df_2020 = weather_km[weather_km['date'].dt.year == 2020]
X = df_2020.drop(columns = ['date','location','rain_sum','snowfall_sum'])

pipe_km = Pipeline([
    ('sc', StandardScaler()),
    ('pca',PCA(n_components=2)),
    ('km', KMeans(n_clusters=2))
])

pipe_km.fit(X)


cluster_centers = pipe_km.named_steps['km'].cluster_centers_
centroids_pca = pipe_km.named_steps['pca'].inverse_transform(cluster_centers)
centroids = pipe_km.named_steps['sc'].inverse_transform(centroids_pca)


centroids_df = pd.DataFrame(
    centroids,
    columns=X.columns
)

In [ ]:
# Starting with n_components=5 explained ~91% of variance, but silhouette score
# improved when reducing to n_components=2. The pipeline reflects the final choice.
var_exp = pipe_km['pca'].explained_variance_ratio_
np.round(var_exp[:8],3)

In [ ]:
# commenting out since it takes about 5 min to run 
X_scaled_pca = pipe_km.named_steps['pca'].transform(X)
cluster_labels = pipe_km.named_steps['km'].labels_
silhouette_score(X_scaled_pca, cluster_labels)

# got a score of 0.29 since it goes from -1 to 1 this is a pretty bad score when using PCA n_components 5 and KMeans(n_clusters = 3)

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(X_scaled_pca[:, 0], X_scaled_pca[:, 1], c=cluster_labels, cmap='PiYG', alpha=0.6)
plt.title('Clusters based on 2020 New York Weather Data')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.tight_layout() 
plt.savefig("../images/Clusters_New_York_weather_2020.png", dpi = 300)
plt.show();

### Time Series K-Means Clustering Model

Before fitting the time-series model, we drop `rain_sum`, `showers_sum`, and
`snowfall_sum`. These were consolidated into `precipitation_total` during
feature engineering, so retaining the individual components would double-count
precipitation's influence in the clustering.

In [ ]:
weather = weather.drop(columns = ['rain_sum', 'showers_sum', 'snowfall_sum'])

In [ ]:
weather['date'] = pd.to_datetime(weather['date'])
weather = weather.sort_values(by = 'date')

##### Forecast Each Feature Through 2030 with Prophet

In [ ]:
# Function to preprocess data, aggregate to monthly, and forecast with Prophet
def preprocess_and_forecast_prophet(data, forecast_end_year):
    """
    Preprocess data to monthly frequency, then forecast each feature per location 
    using Prophet out to forecast_end_year. 
    
    Returns:
        forecasted_data: np.ndarray of shape (n_locations, n_total_months, n_features)
        feature_columns: list of feature names
        all_dates: pd.DatetimeIndex corresponding to the timeline of historical + forecast
    """
    # 1. Ensure 'date' is a proper datetime
    data['date'] = pd.to_datetime(data['date'])

    # 2. Aggregate to monthly averages
    data['month'] = data['date'].dt.to_period('M')
    monthly_data = (
        data.groupby(['latitude', 'longitude', 'month'])
            .mean(numeric_only=True)
            .reset_index()
    )

    # Identify numeric feature columns
    feature_columns = [
        c for c in monthly_data.columns
        if c not in ['latitude', 'longitude', 'month']
    ]

    # Identify all unique locations
    locations = monthly_data[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)

    # Determine the min and max month in the data
    earliest_month = monthly_data['month'].min()
    latest_month = monthly_data['month'].max()
    
    # Convert those to actual timestamps
    earliest_date = earliest_month.to_timestamp()
    latest_date = latest_month.to_timestamp()

        # Build the full date range from earliest historical date up to forecast_end_year
    all_dates = pd.date_range(
        start=earliest_date,
        end=f"{forecast_end_year}-12-01",
        freq='M'
    )

    # Prepare an array to store forecasts: shape (n_locations, len(all_dates), n_features)
    n_locs = len(locations)
    n_timestamps = len(all_dates)
    n_feats = len(feature_columns)
    forecasted_data = np.zeros((n_locs, n_timestamps, n_feats))

    for loc_idx, (lat, lon) in enumerate(locations.values):
        # Extract data for this location
        loc_df = monthly_data[
            (monthly_data['latitude'] == lat) & 
            (monthly_data['longitude'] == lon)
        ].copy()

        # Convert 'month' to datetime for Prophet
        loc_df['ds'] = loc_df['month'].dt.to_timestamp()
        loc_df = loc_df.sort_values('ds')

        # For each feature, fit Prophet on the entire historical data and forecast
        for feat_idx, feat in enumerate(feature_columns):
            # Prepare the DataFrame for Prophet
            prophet_df = loc_df[['ds', feat]].rename(columns={feat: 'y'}).copy()
            prophet_df['y'] = prophet_df['y'].ffill()
            prophet_df['y'] = prophet_df['y'].bfill()

            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False
            )
            model.fit(prophet_df)

            # Create a future dataframe covering all_dates
            future_df = pd.DataFrame({'ds': all_dates})
            forecast = model.predict(future_df)

            # Align predicted 'yhat' with all_dates
            merged_forecast = pd.merge(
                future_df[['ds']], forecast[['ds', 'yhat']],
                on='ds', how='left'
            ).sort_values('ds')
            merged_forecast['yhat'] = merged_forecast['yhat'].ffill()
            merged_forecast['yhat'] = merged_forecast['yhat'].bfill()

            forecasted_data[loc_idx, :, feat_idx] = merged_forecast['yhat'].values

    return forecasted_data, feature_columns, all_dates, locations

In [ ]:
# Define the training period to end in 2023-12 and the data extends to 2024-12
forecasted_data, feature_cols, all_dates, locations = preprocess_and_forecast_prophet(
    data=weather,
    forecast_end_year=2030
)

### Time Series K-Means Clustering Model

In [ ]:
# The shape is (n_locations, total_months, n_features)
n_locs, n_timestamps, n_feats = forecasted_data.shape

# Flatten to (n_locations, n_timestamps * n_features) for the pipeline.
reshaped_data = forecasted_data.reshape(n_locs, n_timestamps * n_feats)

# Define a pipeline for scaling, PCA, and time-series k-means.
# n_jobs=12 requires a CPU with at least 6 cores / 12 threads — reduce if needed.
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=5)),
    ("kmeans", TimeSeriesKMeans(
        n_clusters=2,
        metric="softdtw",
        n_jobs=12,
        verbose=True
    ))
])

pipeline.fit(reshaped_data)
# Cluster labels and silhouette score are extracted in the next cell.

In [ ]:
scaled = pipeline.named_steps['scaler'].transform(reshaped_data)
pca_data = pipeline.named_steps['pca'].transform(scaled)
clusters = pipeline.named_steps['kmeans'].labels_

sil_score = silhouette_score(pca_data, clusters, metric='euclidean')
print("Silhouette Score:", sil_score)

In [ ]:
def cluster_summary(forecasted_data, clusters, feature_names):
    """
    Summarize the average weather characteristics by cluster.

    Args:
        forecasted_data (np.ndarray): shape (n_locs, n_timestamps, n_features)
        clusters (np.ndarray): cluster labels of shape (n_locs,)
        feature_names (list): names of the features in the last dimension of forecasted_data
    
    Returns:
        pd.DataFrame: each row is a cluster, columns = average feature values
    """
    n_clusters = np.max(clusters) + 1
    summaries = []
    for c in range(n_clusters):
        # Indices of locations in cluster c
        idx_in_cluster = np.where(clusters == c)[0]
        
        # Subset the forecasted_data for these locations
        # shape: (n_locations_in_cluster, n_timestamps, n_features)
        data_c = forecasted_data[idx_in_cluster]
        
        # Average over both location dimension and time dimension
        # result shape: (n_features,)
        mean_features = data_c.mean(axis=(0,1))  
        
        summaries.append(mean_features)
    
    summary_df = pd.DataFrame(summaries, columns=feature_names)
    summary_df.index = [f"Cluster {i}" for i in range(n_clusters)]
    return summary_df

cluster_summary(forecasted_data, clusters, feature_cols)

In [ ]:
def plot_clusters_on_map(locations_df, cluster_assignments, title="Clusters on Map", filepath="../images/tskm_clusters.png"):
    """
    Plots each row of `locations_df` colored by its cluster assignment.

    locations_df: pd.DataFrame with columns ['latitude','longitude'], shape=(n_locs,2)
    cluster_assignments: np.ndarray of length n_locs
    """
    plt.figure(figsize=(7,6))
    scatter = plt.scatter(
        x=locations_df['longitude'],
        y=locations_df['latitude'],
        c=cluster_assignments,
        cmap='PiYG',
        alpha=0.7
    )
    plt.legend(['Solar', 'Wind'])
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    plt.savefig(filepath, dpi = 300)
    plt.show();

# Now plot
plot_clusters_on_map(locations, clusters, 'New York State by Optimal Energy Source')

## Energy Load Data

### Load in the Load and Population Data

In [ ]:
load_data_df = pd.read_csv('../data/Newyork_state_load_data.csv')
population_df = pd.read_csv('../data/Annual_Population_Newyork.csv')

### Inspect and Handle the Load Data

In [ ]:
energy_zones_to_counties = {
    'CAPITL': [
        'Albany County', 'Schenectady County', 'Rensselaer County', 'Saratoga County',
        'Columbia County', 'Greene County', 'Washington County'
    ],
    'CENTRL': [
        'Onondaga County', 'Oswego County', 'Cayuga County', 'Cortland County',
        'Tompkins County', 'Madison County', 'Chenango County', 'Broome County'
    ],
    'DUNWOD': ['Rockland County', 'Orange County', 'Putnam County'],
    'GENESE': [
        'Monroe County', 'Genesee County', 'Livingston County', 'Ontario County',
        'Orleans County', 'Wyoming County'
    ],
    'HUD VL': [
        'Dutchess County', 'Ulster County', 'Sullivan County', 'Delaware County',
        'Schoharie County'
    ],
    'LONGIL': ['Nassau County', 'Suffolk County'],
    'MHK VL': [
        'Herkimer County', 'Oneida County', 'Montgomery County', 'Fulton County',
        'Schoharie County', 'Otsego County', 'Chenango County'
    ],
    'MILLWD': ['Westchester County'],
    'N.Y.C.': [
        'New York County', 'Bronx County', 'Queens County', 
        'Kings County', 'Richmond County'
    ],
    'NORTH': [
        'Clinton County', 'Essex County', 'Franklin County', 'Hamilton County',
        'St. Lawrence County', 'Jefferson County', 'Lewis County', 'Warren County'
    ],
    'WEST': [
        'Erie County', 'Niagara County', 'Chautauqua County', 'Cattaraugus County',
        'Allegany County', 'Steuben County', 'Chemung County', 'Tioga County',
        'Wayne County', 'Seneca County', 'Schuyler County', 'Yates County'
    ],
    'N.Y.C._LONGIL': ['Nassau County', 'Suffolk County', 'Bronx County', 'Queens County', 'Kings County']
}


### Load Data

In [ ]:
load_data_df = pd.read_csv('../data/Newyork_state_load_data.csv')

In [ ]:
load_data_df = load_data_df.drop(columns=['Unnamed: 0'])

### Data Exploration

##### *Finding the Zonal-weightage of the each county according to population of each county*

In [ ]:
# Initialize a list to store the results
results = []

# Iterate through each year in the population data
for year in population_df["Year"].unique():
    # Filter data for the current year
    year_data = population_df[population_df["Year"] == year]

    # Calculate zonal weightages
    for zone, counties in energy_zones_to_counties.items():
        # Filter counties in the current zone
        zone_data = year_data[year_data["Geography"].isin(counties)]
        
        # Calculate total population for the zone
        zone_population = zone_data["Population"].sum()
        
        # Calculate the weightage for each county in the zone
        for _, row in zone_data.iterrows():
            weightage = row["Population"] / zone_population if zone_population > 0 else 0
            results.append({"Year": year, "Zone": zone, "County": row["Geography"], "Weightage": weightage})

# Convert the results into a DataFrame
zonal_weightage_df = pd.DataFrame(results)

##### *Finding the Load of each county as per their Weightage*

In [ ]:
# Convert the Date column in load_data_df to datetime format and extract the year
load_data_df["Date"] = pd.to_datetime(load_data_df["Date"])
load_data_df["Year"] = load_data_df["Date"].dt.year

# Merge load data with zonal weightage data on 'Year' and 'Zone'
merged_df = pd.merge(
    load_data_df.rename(columns={"Name": "Zone"}),  # Rename 'Name' to 'Zone' for consistency
    zonal_weightage_df,
    on=["Year", "Zone"],
    how="inner"
)

# Calculate the load for each county
merged_df["County Load"] = merged_df["Load"] * merged_df["Weightage"]

# Select relevant columns for the final output
county_load_df = merged_df[["Date", "County", "County Load"]]


In [ ]:
county_load_df = county_load_df.groupby(["Date",'County'], as_index=False).sum()

We write the unpivoted `county_load_df` to CSV here as an intermediate
checkpoint. The following cell pivots it to wide format, and Cell 80 overwrites
this file with the pivoted version. The final CSV has dates as rows and counties
as columns.

In [ ]:
county_load_df.to_csv('../data/Counties load data.csv')

##### Pivoting the `county_load_df`

In [ ]:
# Pivot the DataFrame to make counties the columns and dates the rows
pivoted_county_load_df = county_load_df.pivot(index="Date", columns="County", values="County Load")

# Reset the index to make Date a column for easier readability
pivoted_county_load_df = pivoted_county_load_df.reset_index()

In [ ]:
pivoted_county_load_df.to_csv('../data/Counties load data.csv')

#### *Forecasting for the Future Years*

In [ ]:
data=pd.read_csv('../data/Counties load data.csv')

In [ ]:
data = data.drop(columns=['Unnamed: 0'])

### Predictive Forecasting

1. [Time Series Advance Analytics techniques](https://www.advancinganalytics.co.uk/blog/2021/06/22/10-incredibly-useful-time-series-forecasting-algorithms)
2. [Prophet Method Details](https://iopscience.iop.org/article/10.1088/1742-6596/2356/1/012002/pdf#:~:text=Finally%2C%20the%20Prophet%20model%20is,the%20electricity%20load%20forecasting%20model)


##### Converting the Date column and handling missing values

In [ ]:
# Convert 'Date' column to datetime format
data['Date'] = pd.to_datetime(data['Date'])

# Fill missing values with forward-fill method
data = data.ffill()

#### EDA

In [ ]:
# Load the dataset 
file_path = '../data/load_data_from_2005-01-01_to_2030-12-31.csv'  
data = pd.read_csv(file_path)

In [ ]:
# Dropping the Unamed: 0
data.drop(columns=['Unnamed: 0'])

In [ ]:
# Preprocessing: Convert Date column to datetime and set as index
data['Date'] = pd.to_datetime(data['Date'])
data = data.set_index('Date')

We sum across all county columns to produce a statewide total. The `iloc[:,
:-1]` slice excludes the last column — this prevents `Total Consumption` from
being included in its own sum if the cell is re-run after the column already
exists.

In [ ]:
# Create Total Consumption column
data['Total Consumption'] = data.iloc[:, :-1].sum(axis=1)

##### Defining the Prophet Forecast Model function

The `prepare_data_for_prophet` and `forecast_county` functions below were also
extracted into `County_Forecasting_ETL.py` for standalone use. They are
reproduced here so the notebook runs end-to-end without requiring the ETL
script to be imported.

In [ ]:
# Function to prepare data for Prophet model
def prepare_data_for_prophet(df, date_col, value_col):
    """
    Prepares a dataframe for Prophet by renaming columns to 'ds' and 'y'.
    Args:
        df: Original dataframe
        date_col: Name of the column containing dates
        value_col: Name of the column containing the values
    Returns:
        Prepared dataframe
    """
    prepared_df = df[[date_col, value_col]].rename(columns={date_col: 'ds', value_col: 'y'})
    return prepared_df

# Function to train and forecast using Prophet
def forecast_county(data, county_name, future_years=2050):
    """
    Trains a Prophet model and forecasts future values for a given county.
    Args:
        data: Original dataframe
        county_name: Name of the county column to forecast
        future_years: Year up to which to forecast
    Returns:
        Dataframe containing forecasted values
    """
    # Prepare data for the selected county
    county_data = prepare_data_for_prophet(data, 'Date', county_name)
    
    # Initialize Prophet model
    model = Prophet()
    model.fit(county_data)
    
    # Create a dataframe for future dates
    last_date = county_data['ds'].max()
    future_dates = model.make_future_dataframe(periods=(future_years - last_date.year) * 365)
    
    # Forecast
    forecast = model.predict(future_dates)
    return forecast[['ds', 'yhat']]

##### Applying Prophet Forecast Model for all the Counties and saving it into the file

##### 1. Seasonality Analysis: Monthly Averages

In [ ]:
monthly_avg = data['Total Consumption'].resample('M').mean()
plt.figure(figsize=(14, 6))
plt.plot(monthly_avg, label='Monthly Average Consumption', color='orange')
plt.title('Monthly Average Energy Consumption (2005-2030)', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Average Energy Consumption (units)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.savefig('../monthly_average_energy_consumption.png')
plt.show()

##### 2. County-Level Analysis: Top Consumer County in Avg Megawatts Per Day Over the Years

In [ ]:
daily_avg = data.iloc[:, :-1].resample('D').mean().mean()
top_consumer_county = daily_avg.idxmax()
plt.figure(figsize=(10, 6))
daily_avg.sort_values(ascending=False).head(5).plot(kind='bar', color='teal')
plt.title(f'Top Consumer County: {top_consumer_county}', fontsize=16)
plt.ylabel('Average Megawatts Per Day', fontsize=14)
plt.xticks(rotation=45)
plt.grid(alpha=0.5)
plt.savefig('../images/top_consumer_county_avg_megawatts_per_day.png')
plt.show()

##### 3. Total Megawatts Per Day for the Whole State Over the Year

In [ ]:
state_yearly_avg = data['Total Consumption'].resample('Y').mean()
plt.figure(figsize=(14, 6))
plt.plot(state_yearly_avg, label='State Yearly Average (Megawatts)', color='purple')
plt.title('Total Megawatts Per Year for the Whole State (2005-2030)', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Yearly Average Energy Consumption (Megawatts)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.savefig('../images/state_yearly_avg_megawatts_per_year.png')
plt.show()

##### 4. Comparing Past and Predicted Trends

In [ ]:
past_monthly_data = data.loc[data.index < '2024-01-01', 'Total Consumption'].resample('M').sum()
future_monthly_data = data.loc[data.index >= '2024-01-01', 'Total Consumption'].resample('M').sum()

plt.figure(figsize=(14, 6))
plt.plot(past_monthly_data, label='Historical Data (2005-2023)', color='blue')
plt.plot(future_monthly_data, label='Predicted Data (2024-2030)', color='green', linestyle='--')
plt.title('Comparison of Past and Predicted Energy Consumption (Monthly)', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Monthly Energy Consumption (units)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.savefig('../images/past_vs_predicted_monthly_energy_consumption.png')
plt.show()

### Findings and Implications
Our present analysis suggests that New York State is in a great position to take advantage of wind and solar farms to produce and store energy to service the whole State's needs. The construction, staffing, and maintenance of these new facilities is projected to produce around 250,000 new jobs and save the State billions in the long term, compared to the present energy infrastructure. As the effects of human-accelerated climate change become more visible, the climatological zones of New York State will muddle as well, with the "solar" zone growing larger.

### Next Steps
Future work would confirm that our proposal accurately assesses and meets New York's energy needs through 2050 and beyond. We do caution, however, that long-term time-series projections tend to be unreliable, though there is a margin of confidence we may able to provide. The New York State Energy Planning Board should proceed to check land use agreements in accordance with our recommendations.